# 📊 Business Insights & Decision Making Dashboard

**BUSINESS INTELLIGENCE** - Transform AI analysis into actionable business decisions

## What this notebook provides:
- 📈 **Key Performance Indicators (KPIs)** - Priority distribution, urgency trends, system impact
- 🎯 **Actionable Recommendations** - Resource allocation, priority management, risk mitigation
- 📋 **Executive Summary** - High-level insights for management
- 🔍 **Deep Dive Analysis** - Detailed breakdowns for technical teams
- 📊 **Dashboard-Ready Data** - Structured for Streamlit visualization

**Prerequisites:** Run `01_sample_data_generation.ipynb` and `02_ai_showcase.ipynb` first

---

## 🎯 Goal: Transform AI insights into business value


In [1]:
# Import required libraries
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json
from datetime import datetime, timedelta
import pandas as pd
import builtins  # Import builtins to access Python's round function

# Import configuration
%run ./config

print("✅ Libraries imported and configuration loaded")
print(f"🎯 Using Unity Catalog: {UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}")
print("📊 Ready for business insights analysis!")


✅ Libraries imported and configuration loaded
🎯 Using Unity Catalog: quickstart_catalog_vkm_external.classify_tickets
📊 Ready for business insights analysis!


# 📊 Data Loading & Validation

Load AI analysis results and validate data quality


In [2]:
# Load AI Analysis Results
print("📊 Loading AI analysis results from Unity Catalog...")

try:
    df_ai_results = spark.table(TABLES["ai_showcase_results"])
    print(f"✅ Loaded {df_ai_results.count()} tickets from: {TABLES['ai_showcase_results']}")
    
    # Display sample data
    print("\n📋 Sample AI Analysis Results:")
    display(df_ai_results.select(
        "ticket_id", 
        "short_description", 
        "ai_priority_classification",
        "action_items",
        "urgency_level"
    ).limit(3))
    
except Exception as e:
    print(f"❌ Error loading data: {e}")
    print("Please ensure you've run the previous notebooks first.")


📊 Loading AI analysis results from Unity Catalog...


✅ Loaded 64 tickets from: quickstart_catalog_vkm_external.classify_tickets.ai_showcase_results

📋 Sample AI Analysis Results:


,ticket_id,short_description,ai_priority_classification,action_items,urgency_level
0,TICKET_01_001,Issue #1-1 - Important,Urgent Priority,investigate,immediately
1,TICKET_01_002,Issue #1-2 - Urgent,Urgent Priority,restore the chat support widget,high
2,TICKET_01_003,Issue #1-3 - Problem,High Priority,figure out what went wrong and need better testing,None


# 📈 Key Performance Indicators (KPIs)

Calculate essential business metrics for decision making


In [3]:
# KPI 1: Priority Distribution Analysis
print("📈 KPI 1: Priority Distribution Analysis")
print("="*50)

priority_distribution = df_ai_results.groupBy("ai_priority_classification").count().orderBy(desc("count"))
priority_distribution.show()

# Calculate priority percentages
total_tickets = df_ai_results.count()
priority_percentages = priority_distribution.withColumn(
    "percentage", 
    round(col("count") / total_tickets * 100, 2)
)

print("\n📊 Priority Distribution Percentages:")
priority_percentages.show()


📈 KPI 1: Priority Distribution Analysis


+--------------------------+-----+
|ai_priority_classification|count|
+--------------------------+-----+
|           Urgent Priority|   38|
|             High Priority|   22|
|           Medium Priority|    4|
+--------------------------+-----+




📊 Priority Distribution Percentages:


+--------------------------+-----+----------+
|ai_priority_classification|count|percentage|
+--------------------------+-----+----------+
|           Urgent Priority|   38|     59.38|
|             High Priority|   22|     34.38|
|           Medium Priority|    4|      6.25|
+--------------------------+-----+----------+



In [4]:
# KPI 2: Urgency Level Analysis
print("📈 KPI 2: Urgency Level Analysis")
print("="*50)

urgency_analysis = df_ai_results.groupBy("urgency_level").count().orderBy(desc("count"))
urgency_analysis.show()

# Calculate urgency percentages
urgency_percentages = urgency_analysis.withColumn(
    "percentage", 
    round(col("count") / total_tickets * 100, 2)
)

print("\n📊 Urgency Distribution Percentages:")
urgency_percentages.show()


📈 KPI 2: Urgency Level Analysis


+--------------------+-----+
|       urgency_level|count|
+--------------------+-----+
|                NULL|   27|
|                high|   25|
|         immediately|    3|
|before the securi...|    3|
|  we're losing sales|    3|
|              urgent|    2|
|         going crazy|    1|
+--------------------+-----+


📊 Urgency Distribution Percentages:


+--------------------+-----+----------+
|       urgency_level|count|percentage|
+--------------------+-----+----------+
|                NULL|   27|     42.19|
|                high|   25|     39.06|
|         immediately|    3|      4.69|
|before the securi...|    3|      4.69|
|  we're losing sales|    3|      4.69|
|              urgent|    2|      3.13|
|         going crazy|    1|      1.56|
+--------------------+-----+----------+



In [5]:
# KPI 3: Affected Systems Analysis
print("📈 KPI 3: Affected Systems Analysis")
print("="*50)

# Extract and count affected systems
systems_analysis = df_ai_results.filter(col("affected_systems").isNotNull())
systems_count = systems_analysis.groupBy("affected_systems").count().orderBy(desc("count"))

print("\n📊 Most Affected Systems:")
systems_count.show(10, truncate=False)

# Calculate system impact percentage
systems_with_data = systems_analysis.count()
if systems_with_data > 0:
    systems_percentages = systems_count.withColumn(
        "percentage", 
        round(col("count") / systems_with_data * 100, 2)
    )
    print("\n📊 System Impact Percentages:")
    systems_percentages.show(10, truncate=False)


📈 KPI 3: Affected Systems Analysis

📊 Most Affected Systems:


+------------------------------------------+-----+
|affected_systems                          |count|
+------------------------------------------+-----+
|the whole system                          |6    |
|chat support widget                       |6    |
|mobile app on iOS devices                 |5    |
|file upload feature and business processes|5    |
|API                                       |4    |
|backup system                             |3    |
|data migration                            |3    |
|email notification system                 |3    |
|reporting dashboard                       |3    |
|our website                               |3    |
+------------------------------------------+-----+
only showing top 10 rows



📊 System Impact Percentages:


+------------------------------------------+-----+----------+
|affected_systems                          |count|percentage|
+------------------------------------------+-----+----------+
|the whole system                          |6    |9.38      |
|chat support widget                       |6    |9.38      |
|mobile app on iOS devices                 |5    |7.81      |
|file upload feature and business processes|5    |7.81      |
|API                                       |4    |6.25      |
|backup system                             |3    |4.69      |
|data migration                            |3    |4.69      |
|email notification system                 |3    |4.69      |
|reporting dashboard                       |3    |4.69      |
|our website                               |3    |4.69      |
+------------------------------------------+-----+----------+
only showing top 10 rows


# 🎯 Business Decision Making

Transform data into actionable business insights


In [6]:
# Business Decision 1: Resource Allocation Recommendations
print("🎯 Business Decision 1: Resource Allocation Recommendations")
print("="*60)

# Calculate resource allocation based on priority and urgency
resource_allocation = df_ai_results.select(
    "ticket_id",
    "short_description",
    "ai_priority_classification",
    "urgency_level",
    "action_items",
    "affected_systems"
).withColumn(
    "resource_priority_score",
    when(col("ai_priority_classification") == "Urgent Priority", 4)
    .when(col("ai_priority_classification") == "High Priority", 3)
    .when(col("ai_priority_classification") == "Medium Priority", 2)
    .otherwise(1)
).withColumn(
    "urgency_score",
    when(col("urgency_level").contains("urgent") | col("urgency_level").contains("asap") | col("urgency_level").contains("high"), 3)
    .when(col("urgency_level").contains("medium") | col("urgency_level").contains("normal"), 2)
    .otherwise(1)
).withColumn(
    "total_priority_score",
    col("resource_priority_score") + col("urgency_score")
).orderBy(desc("total_priority_score"))

print("\n📋 Top Priority Tickets for Resource Allocation:")
display(resource_allocation.select(
    "ticket_id",
    "short_description",
    "ai_priority_classification",
    "urgency_level",
    "total_priority_score",
    "action_items"
).limit(5))


🎯 Business Decision 1: Resource Allocation Recommendations

📋 Top Priority Tickets for Resource Allocation:


,ticket_id,short_description,ai_priority_classification,urgency_level,total_priority_score,action_items
0,TICKET_02_004,Issue #2-4 - Problem,Urgent Priority,high,7,restore the chat support widget
1,TICKET_01_005,Issue #1-5 - Help needed,Urgent Priority,high,7,None
2,TICKET_02_002,Issue #2-2 - Important,Urgent Priority,high,7,restore the chat support widget
3,TICKET_01_002,Issue #1-2 - Urgent,Urgent Priority,high,7,restore the chat support widget
4,TICKET_02_007,Issue #2-7 - Critical,Urgent Priority,high,7,None


In [7]:
# Business Decision 2: Risk Assessment & Mitigation
print("🎯 Business Decision 2: Risk Assessment & Mitigation")
print("="*60)

# Identify high-risk tickets
high_risk_tickets = df_ai_results.filter(
    (col("ai_priority_classification") == "Urgent Priority") |
    (col("urgency_level").contains("urgent")) |
    (col("urgency_level").contains("asap")) |
    (col("urgency_level").contains("high"))
).select(
    "ticket_id",
    "short_description",
    "ai_priority_classification",
    "urgency_level",
    "affected_systems",
    "action_items"
)

print(f"\n⚠️ High-Risk Tickets Identified: {high_risk_tickets.count()}")
print("\n📋 High-Risk Tickets Requiring Immediate Attention:")
display(high_risk_tickets.limit(5))

# Calculate risk distribution
risk_distribution = df_ai_results.withColumn(
    "risk_level",
    when(
        (col("ai_priority_classification") == "Urgent Priority") |
        (col("urgency_level").contains("urgent")) |
        (col("urgency_level").contains("asap")),
        "High Risk"
    ).when(
        (col("ai_priority_classification") == "High Priority") |
        (col("urgency_level").contains("high")),
        "Medium Risk"
    ).otherwise("Low Risk")
).groupBy("risk_level").count().orderBy(desc("count"))

print("\n📊 Risk Distribution:")
risk_distribution.show()


🎯 Business Decision 2: Risk Assessment & Mitigation



⚠️ High-Risk Tickets Identified: 44

📋 High-Risk Tickets Requiring Immediate Attention:


,ticket_id,short_description,ai_priority_classification,urgency_level,affected_systems,action_items
0,TICKET_01_001,Issue #1-1 - Important,Urgent Priority,immediately,data migration,investigate
1,TICKET_01_002,Issue #1-2 - Urgent,Urgent Priority,high,chat support widget,restore the chat support widget
2,TICKET_01_005,Issue #1-5 - Help needed,Urgent Priority,high,mobile app on iOS devices,None
3,TICKET_02_001,Issue #2-1 - Problem,Urgent Priority,before the security team gets involved,authentication,need to fix this
4,TICKET_02_002,Issue #2-2 - Important,Urgent Priority,high,chat support widget,restore the chat support widget



📊 Risk Distribution:


+-----------+-----+
| risk_level|count|
+-----------+-----+
|  High Risk|   38|
|Medium Risk|   22|
|   Low Risk|    4|
+-----------+-----+



In [8]:
# Business Decision 3: System Health & Maintenance Planning
print("🎯 Business Decision 3: System Health & Maintenance Planning")
print("="*60)

# Analyze system health based on ticket patterns
system_health = df_ai_results.filter(col("affected_systems").isNotNull())
system_health_analysis = system_health.groupBy("affected_systems").agg(
    count("ticket_id").alias("ticket_count"),
    countDistinct("ticket_id").alias("unique_tickets"),
    sum(when(col("ai_priority_classification") == "Urgent Priority", 1).otherwise(0)).alias("urgent_count"),
    sum(when(col("ai_priority_classification") == "High Priority", 1).otherwise(0)).alias("high_priority_count")
).withColumn(
    "urgent_percentage",
    round(col("urgent_count") / col("ticket_count") * 100, 2)
).withColumn(
    "health_score",
    when(col("urgent_percentage") > 50, "Critical")
    .when(col("urgent_percentage") > 25, "Poor")
    .when(col("urgent_percentage") > 10, "Fair")
    .otherwise("Good")
).orderBy(desc("urgent_percentage"))

print("\n📊 System Health Analysis:")
display(system_health_analysis.select(
    "affected_systems",
    "ticket_count",
    "urgent_count",
    "urgent_percentage",
    "health_score"
).limit(10))


🎯 Business Decision 3: System Health & Maintenance Planning

📊 System Health Analysis:


,affected_systems,ticket_count,urgent_count,urgent_percentage,health_score
0,mobile app on iOS devices,5,5,100.0,Critical
1,backup system,3,3,100.0,Critical
2,chat support widget on the website,1,1,100.0,Critical
3,password reset functionality,2,2,100.0,Critical
4,chat support widget,6,6,100.0,Critical
5,email notification system,3,3,100.0,Critical
6,login system,2,2,100.0,Critical
7,authentication,2,2,100.0,Critical
8,the whole system,6,6,100.0,Critical
9,data migration system,1,1,100.0,Critical


# 📋 Executive Summary & Recommendations

Generate high-level insights for management decision making


In [9]:
# Executive Summary Generation
print("📋 Executive Summary & Recommendations")
print("="*50)

# Calculate executive summary metrics
executive_metrics = df_ai_results.agg(
    count("ticket_id").alias("total_tickets"),
    sum(when(col("ai_priority_classification") == "Urgent Priority", 1).otherwise(0)).alias("urgent_tickets"),
    sum(when(col("ai_priority_classification") == "High Priority", 1).otherwise(0)).alias("high_priority_tickets"),
    sum(when(col("urgency_level").contains("urgent") | col("urgency_level").contains("asap"), 1).otherwise(0)).alias("urgent_urgency_tickets"),
    countDistinct("affected_systems").alias("unique_systems_affected")
).collect()[0]

# Convert to Python values for calculations
total_tickets = int(executive_metrics["total_tickets"])
urgent_tickets = int(executive_metrics["urgent_tickets"])
high_priority_tickets = int(executive_metrics["high_priority_tickets"])
urgent_urgency_tickets = int(executive_metrics["urgent_urgency_tickets"])
unique_systems = int(executive_metrics["unique_systems_affected"])

print(f"\n📊 EXECUTIVE SUMMARY METRICS:")
print(f"Total Tickets Analyzed: {total_tickets}")
print(f"Urgent Priority Tickets: {urgent_tickets} ({builtins.round(urgent_tickets/total_tickets*100, 1)}%)")
print(f"High Priority Tickets: {high_priority_tickets} ({builtins.round(high_priority_tickets/total_tickets*100, 1)}%)")
print(f"Urgent Urgency Level: {urgent_urgency_tickets} ({builtins.round(urgent_urgency_tickets/total_tickets*100, 1)}%)")
print(f"Unique Systems Affected: {unique_systems}")

# Generate recommendations
print(f"\n🎯 KEY RECOMMENDATIONS:")
print(f"1. IMMEDIATE ACTION: {urgent_tickets} urgent tickets require immediate attention")
print(f"2. RESOURCE ALLOCATION: Focus {builtins.round(urgent_tickets/total_tickets*100, 1)}% of resources on urgent issues")
print(f"3. SYSTEM MONITORING: {unique_systems} different systems need monitoring")
print(f"4. RISK MITIGATION: {urgent_urgency_tickets} tickets have urgent urgency levels")

if urgent_tickets > total_tickets * 0.3:
    print(f"5. ⚠️  HIGH ALERT: {builtins.round(urgent_tickets/total_tickets*100, 1)}% urgent tickets - consider emergency response")
elif urgent_tickets > total_tickets * 0.1:
    print(f"5. ⚠️  MODERATE ALERT: {builtins.round(urgent_tickets/total_tickets*100, 1)}% urgent tickets - increase monitoring")
else:
    print(f"5. ✅ NORMAL OPERATIONS: {builtins.round(urgent_tickets/total_tickets*100, 1)}% urgent tickets - maintain current processes")


📋 Executive Summary & Recommendations



📊 EXECUTIVE SUMMARY METRICS:
Total Tickets Analyzed: 64
Urgent Priority Tickets: 38 (59.4%)
High Priority Tickets: 22 (34.4%)
Urgent Urgency Level: 2 (3.1%)
Unique Systems Affected: 26

🎯 KEY RECOMMENDATIONS:
1. IMMEDIATE ACTION: 38 urgent tickets require immediate attention
2. RESOURCE ALLOCATION: Focus 59.4% of resources on urgent issues
3. SYSTEM MONITORING: 26 different systems need monitoring
4. RISK MITIGATION: 2 tickets have urgent urgency levels
5. ⚠️  HIGH ALERT: 59.4% urgent tickets - consider emergency response


# 📊 Dashboard-Ready Data Preparation

Prepare structured data for Streamlit dashboard visualization


In [10]:
# Prepare Dashboard Data - Priority Distribution
print("📊 Preparing Dashboard Data - Priority Distribution")

dashboard_priority = priority_percentages.select(
    col("ai_priority_classification").alias("priority_level"),
    col("count").alias("ticket_count"),
    col("percentage").alias("percentage")
).withColumn(
    "chart_color",
    when(col("priority_level") == "Urgent Priority", "#FF4444")
    .when(col("priority_level") == "High Priority", "#FF8800")
    .when(col("priority_level") == "Medium Priority", "#FFAA00")
    .otherwise("#00AA00")
)

print("✅ Priority distribution data prepared for dashboard")
dashboard_priority.show()


📊 Preparing Dashboard Data - Priority Distribution
✅ Priority distribution data prepared for dashboard


+---------------+------------+----------+-----------+
| priority_level|ticket_count|percentage|chart_color|
+---------------+------------+----------+-----------+
|Urgent Priority|          38|     59.38|    #FF4444|
|  High Priority|          22|     34.38|    #FF8800|
|Medium Priority|           4|      6.25|    #FFAA00|
+---------------+------------+----------+-----------+



In [11]:
# Prepare Dashboard Data - System Health
print("📊 Preparing Dashboard Data - System Health")

dashboard_system_health = system_health_analysis.select(
    col("affected_systems").alias("system_name"),
    col("ticket_count").alias("total_tickets"),
    col("urgent_count").alias("urgent_tickets"),
    col("urgent_percentage").alias("urgent_percentage"),
    col("health_score").alias("health_status")
).withColumn(
    "health_color",
    when(col("health_status") == "Critical", "#FF0000")
    .when(col("health_status") == "Poor", "#FF8800")
    .when(col("health_status") == "Fair", "#FFAA00")
    .otherwise("#00AA00")
)

print("✅ System health data prepared for dashboard")
dashboard_system_health.show()


📊 Preparing Dashboard Data - System Health
✅ System health data prepared for dashboard


+--------------------+-------------+--------------+-----------------+-------------+------------+
|         system_name|total_tickets|urgent_tickets|urgent_percentage|health_status|health_color|
+--------------------+-------------+--------------+-----------------+-------------+------------+
|      authentication|            2|             2|            100.0|     Critical|     #FF0000|
|        login system|            2|             2|            100.0|     Critical|     #FF0000|
|      data migration|            3|             3|            100.0|     Critical|     #FF0000|
|mobile app on iOS...|            5|             5|            100.0|     Critical|     #FF0000|
|email notificatio...|            3|             3|            100.0|     Critical|     #FF0000|
|       backup system|            3|             3|            100.0|     Critical|     #FF0000|
|         our website|            3|             3|            100.0|     Critical|     #FF0000|
|our authenticatio...|        

In [12]:
# Prepare Dashboard Data - Resource Allocation
print("📊 Preparing Dashboard Data - Resource Allocation")

dashboard_resource_allocation = resource_allocation.select(
    col("ticket_id").alias("ticket_id"),
    col("short_description").alias("description"),
    col("ai_priority_classification").alias("priority"),
    col("urgency_level").alias("urgency"),
    col("total_priority_score").alias("priority_score"),
    col("action_items").alias("action_items"),
    col("affected_systems").alias("affected_systems")
).withColumn(
    "resource_urgency",
    when(col("priority_score") >= 6, "Immediate")
    .when(col("priority_score") >= 4, "High")
    .when(col("priority_score") >= 2, "Medium")
    .otherwise("Low")
)

print("✅ Resource allocation data prepared for dashboard")
dashboard_resource_allocation.show(5)


📊 Preparing Dashboard Data - Resource Allocation
✅ Resource allocation data prepared for dashboard


+-------------+--------------------+---------------+-------+--------------+--------------------+--------------------+----------------+
|    ticket_id|         description|       priority|urgency|priority_score|        action_items|    affected_systems|resource_urgency|
+-------------+--------------------+---------------+-------+--------------+--------------------+--------------------+----------------+
|TICKET_01_005|Issue #1-5 - Help...|Urgent Priority|   high|             7|                NULL|mobile app on iOS...|       Immediate|
|TICKET_02_007|Issue #2-7 - Crit...|Urgent Priority|   high|             7|                NULL|mobile app on iOS...|       Immediate|
|TICKET_02_002|Issue #2-2 - Impo...|Urgent Priority|   high|             7|restore the chat ...| chat support widget|       Immediate|
|TICKET_01_002| Issue #1-2 - Urgent|Urgent Priority|   high|             7|restore the chat ...| chat support widget|       Immediate|
|TICKET_02_004|Issue #2-4 - Problem|Urgent Priority|   

In [13]:
# Save Dashboard Data to Unity Catalog
print("💾 Saving Dashboard Data to Unity Catalog...")

# Save priority distribution
dashboard_priority.write.format("delta").mode("overwrite").saveAsTable(
    f"{UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}.dashboard_priority_distribution"
)
print("✅ Priority distribution saved")

# Save system health
dashboard_system_health.write.format("delta").mode("overwrite").saveAsTable(
    f"{UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}.dashboard_system_health"
)
print("✅ System health data saved")

# Save resource allocation
dashboard_resource_allocation.write.format("delta").mode("overwrite").saveAsTable(
    f"{UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}.dashboard_resource_allocation"
)
print("✅ Resource allocation data saved")

print(f"\n📊 All dashboard data saved to: {UNITY_CATALOG['catalog_name']}.{UNITY_CATALOG['schema_name']}")


💾 Saving Dashboard Data to Unity Catalog...


✅ Priority distribution saved


✅ System health data saved


✅ Resource allocation data saved

📊 All dashboard data saved to: quickstart_catalog_vkm_external.classify_tickets


# 🎉 Business Insights Summary

**Complete business intelligence analysis ready for dashboard visualization**


In [14]:
# Final Summary
print("🎉 BUSINESS INSIGHTS ANALYSIS COMPLETED!")
print("="*60)
print(f"📊 Total tickets analyzed: {total_tickets}")
print(f"🎯 KPIs calculated: Priority distribution, urgency analysis, system health")
print(f"💡 Business recommendations generated")
print(f"📋 Executive summary prepared")
print(f"📊 Dashboard data saved to Unity Catalog")
print(f"\n🚀 Ready for Streamlit dashboard implementation!")
print("="*60)
print("✅ Priority distribution analysis")
print("✅ Urgency level assessment")
print("✅ System health monitoring")
print("✅ Resource allocation recommendations")
print("✅ Risk assessment & mitigation")
print("✅ Executive summary & KPIs")
print("✅ Dashboard-ready data preparation")
print("="*60)


🎉 BUSINESS INSIGHTS ANALYSIS COMPLETED!
📊 Total tickets analyzed: 64
🎯 KPIs calculated: Priority distribution, urgency analysis, system health
💡 Business recommendations generated
📋 Executive summary prepared
📊 Dashboard data saved to Unity Catalog

🚀 Ready for Streamlit dashboard implementation!
✅ Priority distribution analysis
✅ Urgency level assessment
✅ System health monitoring
✅ Resource allocation recommendations
✅ Risk assessment & mitigation
✅ Executive summary & KPIs
✅ Dashboard-ready data preparation
